# Day 3: Prompt Templates

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** stop gluing strings together. Write a prompt once with named holes, fill it many times, and dodge the brace trap that catches everyone who puts JSON in a prompt.

In [4]:
import time

from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter

load_dotenv()

model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free", temperature=0)

## 1. Building a prompt by gluing strings

Summarise a customer email, built by hand:

In [ ]:
email = "The parcel arrived three days late and the box was crushed."

prompt_text = "Summarise this email in one sentence: " + email

model.invoke(prompt_text).text

'The parcel arrived three days late and was delivered in a crushed box.'

Works, but doesn't scale. Add a second variable or a system message and you're gluing strings in five places, scattered through your code instead of living in one.

## 2. Write the prompt once with ChatPromptTemplate

A template is the prompt with named holes in it.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

summariser = ChatPromptTemplate.from_messages([
    ("system", "You summarise customer emails for a support team."),
    ("user", "Summarise this email in one sentence:\n\n{email}"),
])

`{email}` is the hole, nothing filled in yet, no model called.

`("system", ...)` and `("user", ...)` are shorthand for the `SystemMessage`/`HumanMessage` from Day 2.

## 3. Fill the template and inspect the messages

In [7]:
summariser

ChatPromptTemplate(input_variables=['email'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You summarise customer emails for a support team.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['email'], input_types={}, partial_variables={}, template='Summarise this email in one sentence:\n\n{email}'), additional_kwargs={})])

In [8]:
filled = summariser.invoke({"email": email})

filled.messages

[SystemMessage(content='You summarise customer emails for a support team.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Summarise this email in one sentence:\n\nThe parcel arrived three days late and the box was crushed.', additional_kwargs={}, response_metadata={})]

The same message list from Day 2, the template just wrote it for you.

**A template is not a new kind of thing.** It's a factory that produces a message list, and you can always look at what it produced.

In [9]:
model.invoke(filled).text

'The customer reports that their parcel arrived three days late and the box was crushed.'

In [10]:
from langchain.messages import HumanMessage, SystemMessage

model.invoke([SystemMessage(content='You summarise customer emails for a support team.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Summarise this email in one sentence:\n\nThe parcel arrived three days late and the box was crushed.', additional_kwargs={}, response_metadata={})]).text

'The customer reported that their parcel arrived three days late and the box was crushed.'

## 4. input_variables: what the template expects

In [11]:
summariser.input_variables

['email']

Useful when a prompt has grown and you've lost track of what it expects.

## 5. A template with two variables

Templates get useful when there is more than one variable.

In [ ]:
replier = ChatPromptTemplate.from_messages([
    ("system", """You write replies for a customer support team.
Write in a {tone} tone. Keep it under 60 words.
Never promise a refund."""),
    ("user", "Reply to this email:

{email}"),
])

In [13]:
replier

ChatPromptTemplate(input_variables=['email', 'tone'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['tone'], input_types={}, partial_variables={}, template='You write replies for a customer support team.\nWrite in a {tone} tone. Keep it under 60 words.\nNever promise a refund.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['email'], input_types={}, partial_variables={}, template='Reply to this email:\n\n{email}'), additional_kwargs={})])

In [17]:
replier.invoke({"tone": "warm", "email": email})

ChatPromptValue(messages=[SystemMessage(content='You write replies for a customer support team.\nWrite in a warm tone. Keep it under 60 words.\nNever promise a refund.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Reply to this email:\n\nThe parcel arrived three days late and the box was crushed.', additional_kwargs={}, response_metadata={})])

In [14]:
time.sleep(4)

model.invoke(replier.invoke({"tone": "warm", "email": email})).text

"I'm so sorry to hear your parcel arrived late and damaged. That's absolutely not the experience we want for you. I've reported this to our shipping team and will follow up with you directly about next steps. Thank you for your patience."

In [18]:
time.sleep(4)

model.invoke(replier.invoke({"tone": "formal", "email": email})).text

"We sincerely apologize for the late delivery and damaged packaging. This falls short of our service standards. We've forwarded your concern to our logistics team to prevent future occurrences. Please reply with photos of the damaged box and contents so we can process a replacement promptly."

One template, two tones, no string gluing.

*Never promise a refund* lives in the **system** half. Policies belong in the template, written once, not retyped at every call site.

## 6. A missing variable raises KeyError

In [21]:
# replier.invoke({"email": email})

In [22]:
try:
    replier.invoke({"email": email})
except KeyError as problem:
    print(problem)

"Input to ChatPromptTemplate is missing variables {'tone'}.  Expected: ['email', 'tone'] Received: ['email']\nNote: if you intended {tone} to be part of the string and not a variable, please escape it with double curly braces like: '{{tone}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "


It refused rather than sending a broken prompt. A hand-built string would have quietly produced `in a  tone` and cost you an hour of confusion. Loud failures are cheaper than quiet ones.

## 7. Literal braces: why JSON in a prompt breaks

Braces mean "hole". What happens when the prompt genuinely needs one, like JSON?

In [23]:
trap = ChatPromptTemplate.from_messages([
    ("system", 'Reply as JSON like {"airline": "Zephyr"}'),
    ("user", "{complaint}"),
])

trap.input_variables

['"airline"', 'complaint']

Two variables now, one of them `"airline"`. The template read the JSON example as a hole to fill.

In [24]:
try:
    trap.invoke({"complaint": "My flight was cancelled."})
except KeyError as problem:
    print(problem)

'Input to ChatPromptTemplate is missing variables {\'"airline"\'}.  Expected: [\'"airline"\', \'complaint\'] Received: [\'complaint\']\nNote: if you intended {"airline"} to be part of the string and not a variable, please escape it with double curly braces like: \'{{"airline"}}\'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT '


The fix is to double every literal brace.

In [25]:
fixed = ChatPromptTemplate.from_messages([
    ("system", 'Reply as JSON like {{"airline": "Zephyr"}}'),
    ("user", "{complaint}"),
])

fixed.input_variables

['complaint']

In [27]:
fixed.invoke({"complaint": "My flight was cancelled."})

ChatPromptValue(messages=[SystemMessage(content='Reply as JSON like {"airline": "Zephyr"}', additional_kwargs={}, response_metadata={}), HumanMessage(content='My flight was cancelled.', additional_kwargs={}, response_metadata={})])

`{{` and `}}` produce one literal brace each. This catches almost everybody once, now it's caught you here, for free.

## 8. Pinning a value with partial

`partial` fills one hole now and hands back a template with the rest still open.

In [ ]:
warm_replier = replier.partial(tone="warm")

warm_replier

In [30]:
time.sleep(4)

model.invoke(warm_replier.invoke({"email": email})).text

"I'm so sorry to hear your parcel arrived late and damaged. That's really frustrating. Could you please share your order number and photos of the box? We'll investigate with the carrier and make this right for you. Thank you for your patience."

`tone` is gone from what you must supply: one template, two configurations, simpler calling code.

This is how a team gets a prompt pre-configured for its use case, without copying it.

## 9. MessagesPlaceholder for chat history

A hole does not have to be a string. `MessagesPlaceholder` reserves a slot for
a **list of messages**.

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a concise travel assistant."),
    MessagesPlaceholder("history"),
    ("user", "{question}"),
])

In [32]:
chat_prompt

ChatPromptTemplate(input_variables=['history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[langchai

In [33]:
from langchain.messages import HumanMessage, AIMessage

past = [
    HumanMessage("I am flying to Delhi on Friday."),
    AIMessage("Noted, Delhi on Friday."),
]

built = chat_prompt.invoke({"history": past, "question": "What day did I say?"})

built.messages

[SystemMessage(content='You are a concise travel assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I am flying to Delhi on Friday.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Noted, Delhi on Friday.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What day did I say?', additional_kwargs={}, response_metadata={})]

Four messages: system, the two you passed in, and the new question. The placeholder expanded into the middle of the list.

In [34]:
time.sleep(4)

model.invoke(built).text

'You said Friday.'

This is the exact shape of a chatbot prompt, why Day 6 fits neatly on top of today: history in the placeholder, fixed instructions in the system message, never tangled.

## 10. Run one template over many inputs

In [35]:
emails = [
    "The parcel arrived three days late and the box was crushed.",
    "I was charged twice for the same order.",
    "Your app logged me out and will not let me back in.",
]

for e in emails:
    time.sleep(4)
    reply = model.invoke(warm_replier.invoke({"email": e}))
    print(reply.text)
    print("-" * 60)

I'm so sorry to hear your parcel arrived late and damaged. That's definitely not the experience we want for you. I've reported this to our shipping team and will follow up with you directly about next steps. Thank you for your patience.
------------------------------------------------------------
I'm sorry to hear you were charged twice for the same order. That's definitely not what should happen. I've flagged this with our billing team right away, and they'll investigate and resolve it as quickly as possible. You should hear back from us within 24 hours. Thank you for your patience.
------------------------------------------------------------
We're sorry you're having trouble logging back in! This can happen occasionally. Please try closing the app completely and reopening it, then signing in again. If that doesn't work, let us know and we'll help you get back in right away.
------------------------------------------------------------


Three replies from one template. Change the policy once and all three change.

## 11. from_template, for prompts with no system message

In [36]:
quick = ChatPromptTemplate.from_template("Translate to French: {text}")

quick.invoke({"text": "good morning"}).messages

[HumanMessage(content='Translate to French: good morning', additional_kwargs={}, response_metadata={})]

In [37]:
quick

ChatPromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='Translate to French: {text}'), additional_kwargs={})])

One `HumanMessage`, no system message. Convenient for small jobs; reach for `from_messages` as soon as there's an instruction worth stating.